# Statistik-Labor Testat – NANUAG-Gesamtprojekt (Version 1)
**Wintersemester 2026/2027 – Gruppe 02**

***Hinweise:***
- Die Variablen unten sind bereits auf **Ihre Gruppe (02)** voreingestellt.
- Bitte überprüfen Sie Ihre Resultate laufend mit `Teilergebnisse_G02_V01.txt`.
- **Es wird dreimal abgegeben**, jeweils zusammen mit der zugehörigen Präsentation:
  Zwischenabgabe 1, Zwischenabgabe 2 und Endabgabe. Jede Abgabe besteht aus diesem (wachsenden)
  Notebook **und** einem PDF-Export davon, in Moodle.
  Dateiname je Abgabe: `Projekt_Gruppe02_<Nachnamen>_Zwischenabgabe1.ipynb`
  bzw. `..._Zwischenabgabe2.ipynb` bzw. `..._Endabgabe.ipynb`.
- Konkrete Wochentage/Uhrzeiten werden noch bekanntgegeben; siehe Terminplan für die Kalenderwochen.
- Geben Sie **niemals** die vollständige Tabelle mit 100.000 Zeilen aus — nutzen Sie `.head()`,
  `.sample()`, Aggregationen und Diagramme.
- Die vollständige Aufgabenstellung mit allen Teilaufgaben finden Sie in
  `Aufgabenstellung_Gruppe02.pdf`. Dieses Notebook ist das Arbeitsgerüst.

In [ ]:
GRUPPENNUMMER = 2   # bereits auf Ihre Gruppe voreingestellt
VERSION = 1         # dieses Semester fuer alle Gruppen: Version 1

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

g = f"{GRUPPENNUMMER:02d}"
v = f"{VERSION:02d}"
pfad = "./"
print("Arbeite mit Dateien aus:", pfad)

---
# Teil 1 – Personaldaten der NANUAG

## a) Daten einlesen

In [ ]:
roh= pd.read_csv("nanuag_mitarbeiter_G02_V01.csv",sep=";")
#Anzahl Zeilen beim raw input
anzahl_start = len(daten)
daten = roh.copy()
daten.head()

## b) Datenbereinigung
Dokumentieren Sie Ihre Regeln in einer Tabelle (z. B. als DataFrame).

In [ ]:
#Zählen leerer Felder in der Tabelle
#print(daten.isna().sum())
#Ergebnis: 0


#Leerstellen
#wandelt  Felder mit "", werden von isna nicht erkannt, in NaN umwandeln - NA = fehlender Wert, Ausdruck sucht nach "beliebig vielen Leerzeichen"
# "von Anfang Leerzeichen bis Ende Leerzeichen"
daten = daten.replace(r"^\s*$", pd.NA, regex=True)
leerstellen = daten.isna().any(axis=1)

print(leerstellen.sum(), "Zeilen, welche mind. ein leeres Feld haben")
#print(daten[leerstellen])

daten = daten[~leerstellen]


#Löschen Falscher Geburtsdatum Formate
daten["Geburtsdatum"] = pd.to_datetime(daten["Geburtsdatum"], format="%d.%m.%Y", errors="coerce") #Felder in Geburtsdatum mit einem falschen Format werden mit NaN markiert

fehler = daten["Geburtsdatum"].isna()

print(fehler.sum(), "Geburtsdaten welche nicht im Format Tag.Monat.Jahr sind")

#print(daten[fehler])
daten = daten.dropna(subset=["Geburtsdatum"]) #Zeilen wo es mit Nan markierte Felder gibt werden gelöscht

#Löschen von Geburtsdaten welche in der Zukunft liegen oder zu weit in der Vergangenheit
geburtstag_zukunft_vergangenheit = (daten["Geburtsdatum"] > pd.Timestamp.today()) | (daten["Geburtsdatum"] < pd.Timestamp("1900-01-01"))
#print(daten[geburtstag_zukunft_vergangenheit])

print(geburtstag_zukunft_vergangenheit.sum(), "Geburtsdaten welche über dem aktuellen Datum sind oder unter 01.01.1900")

#daten = daten[~geburtstag_zukunft_vergangenheit] #~ steht für Negation. Also alle Werte die True waren wurden False


#Löschen von Alterranges welche unter 16 und über 70 sind via "datetime"
min_alter = 16
max_alter = 70
stichtag_jahr = 2026

geburtsjahr = daten["Geburtsdatum"].dt.year
unplausibel_alter = ((stichtag_jahr - geburtsjahr) < min_alter) | ((stichtag_jahr - geburtsjahr) > max_alter)

print(unplausibel_alter.sum(), "Personen unter", min_alter, "oder über", max_alter, "Jahre")

daten = daten[~unplausibel_alter]

#Löschen Doppelter Personalnummern
anzahl_dopplungen = daten["Personalnummer"].duplicated() #anzahl_dopplungen wird ein series aus True und False. True für alle dopellten Werte außer den ersten

print(anzahl_dopplungen.sum(), "Personalnummern, welche mehr als 1 Mal vorkommen") #ich zähle die anzahl der True Werte

daten = daten.drop_duplicates(subset="Personalnummer") #entfernt alle doppelten Felder außer das erste

#Prüfen ob Gehalt eine *Zahl* ist?
#print(daten["Jahresgehalt"].dtype)
#Ergebnis: float64 also Zahl


# Unplausible Gehälter (Rahmen: negative Zahlen oder 0)
neg_gehalt = (daten["Jahresgehalt"]) <=  0

print(neg_gehalt.sum(), "Jahresgehälter mit negativem Vorzeichen oder gleich 0")
#print(daten[neg_gehalt])

daten = daten[~neg_gehalt]


#Keine Ahnung wie ich Zufallszeichen-ketten in Namen überprüfen soll

#Bade:
#Nur nach Buchstaben via isalpha() prüfen
ungueltiger_vorname = ~daten["Vorname"].str.isalpha()
ungueltiger_nachname = ~daten["Nachname"].str.isalpha()
ungueltige_namen = ungueltiger_vorname | ungueltiger_nachname

print(ungueltige_namen.sum(), "Zeilen mit ungültigem Namenszeichen (nur Buchstaben)")
#print(daten[ungueltige_namen][["Vorname", "Nachname"]])

daten = daten[~ungueltige_namen]

gehalt_bereinigt = daten.copy()
gehalt_bereinigt = gehalt_bereinigt.sort_values("Jahresgehalt")
Q1 = gehalt_bereinigt["Jahresgehalt"].quantile(0.25)
Q3 = gehalt_bereinigt["Jahresgehalt"].quantile(0.75)
IQR = Q3 - Q1
untergrenze = Q1 - 1.5*IQR
obergrenze = Q3 + 1.5*IQR
gehalts_ausreißer = (gehalt_bereinigt["Jahresgehalt"] < untergrenze) | (gehalt_bereinigt["Jahresgehalt"] > obergrenze)
print(gehalts_ausreißer.sum(), "Ausreißer nach folgender Regel: [Q1 − 1,5 · IQR, Q3 + 1,5 · IQR]")

gehalt_bereinigt = gehalt_bereinigt[~gehalts_ausreißer]




In [ ]:
#Protokolltabelle zur Datenbereinigung

protokoll = pd.DataFrame({
    "Regel (in dieser Reihenfolge angewendet)": [
        "Leeres Feld",
        "Geburtsdatum nicht im Format TT.MM.JJJJ",
        "Alter < 16 oder > 70 (zum 31.12.2026, inkl. Zukunftsdaten)",
        "Doppelte Personalnummer",
        "Jahresgehalt <= 0",
        "Ungültige Zeichen im Namen",
        "Gehaltsausreißer (IQR)"
    ],
    "Betroffene Datensätze": [
        leerstellen.sum(),
        fehler.sum(),
        unplausibel_alter.sum(),
        anzahl_dopplungen.sum(),
        neg_gehalt.sum(),
        ungueltige_namen.sum(),
        gehalts_ausreißer.sum()
    ]
})

#Output ohne Zeilennummern
print(protokoll.to_string(index=False))
print("Datensätze vorher:", anzahl_start, "| nachher:", len(gehalt_bereinigt))

## c) Alter bestimmen (Stichtag 31.12.2026)

In [ ]:
#Rahmenbedingungen für Alter: 16-70 Jahre, Stichtag 2026
min_alter = 16
max_alter = 70
stichtag_jahr = 2026

#Logik zur Berechnung des Alters: Stichtagjahr - Geburtsjahr
daten["Alter"] = stichtag_jahr - daten["Geburtsdatum"].dt.year

#Überprüfung ob Bereinigung funktioniert hat
print(unplausibel_alter.sum(), "Personen unter", min_alter, "oder über", max_alter)

#Output
print("Jüngste Person:", daten["Alter"].min(), "Jahre")
print("Älteste Person:", daten["Alter"].max(), "Jahre")
print(daten[["Geburtsdatum", "Alter"]].head(10))

## d) Explorative Analyse (Säulendiagramme, Tortendiagramme)

In [ ]:
#Alter zum Stichtag 31.12.2026
#Daten zur Überprüfung des ersten Diagramms
#Funktionen wie fig, legend kommen aus der matplotlib
gehalt_bereinigt["Alter"] = 2026 - gehalt_bereinigt["Geburtsdatum"].dt.year


gehalt_bereinigt["Altersklasse"] = pd.cut(
    gehalt_bereinigt["Alter"],
    bins=[0, 20, 30, 40, 50, 200],
    labels=["≤20", "21–30", "31–40", "41–50", ">50"]
)

#print(gehalt_bereinigt[["Geburtsdatum", "Alter", "Altersklasse"]].head(10))
print(gehalt_bereinigt["Altersklasse"].value_counts().sort_index())

tabelle_alter = pd.crosstab(gehalt_bereinigt["Altersklasse"], gehalt_bereinigt["Anrede"])

farben = {"Frau": "#E4572E", "Herr": "#2E86AB", "Divers": "#76B041"}
farben_liste = [farben[spalte] for spalte in tabelle_alter.columns]

ax = tabelle_alter.plot(kind="bar", figsize=(10, 6), color=farben_liste, width=0.8, edgecolor="white")

# Zahlen über die Säulen schreiben
for container in ax.containers:
    ax.bar_label(container, fmt="{:,.0f}".format, fontsize=8, padding=2)

ax.set_title("Mitarbeiter nach Altersklasse und Anrede", fontsize=14, fontweight="bold")
ax.set_xlabel("Altersklasse")
ax.set_ylabel("Anzahl Mitarbeiter")
ax.tick_params(axis="x", rotation=0)

# Aufgeräumter Look: nur horizontale Hilfslinien, kein oberer/rechter Rahmen
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)

ax.legend(title="Anrede", frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
altersklassen = ["≤20", "21–30", "31–40", "41–50", ">50"]
farben_klassen = ["#CFE3F1", "#9CC5E0", "#5FA2CC", "#2E7BAE", "#1B4F72"]

fig, achsen = plt.subplots(1, 2, figsize=(11, 5))

for ax, anrede in zip(achsen, ["Frau", "Herr"]):
    teil = gehalt_bereinigt[gehalt_bereinigt["Anrede"] == anrede]
    anzahl = teil["Altersklasse"].value_counts().reindex(altersklassen)
    print(anrede, anzahl.to_dict())

    ax.pie(
        anzahl,
        labels=altersklassen,
        colors=farben_klassen,
        autopct="%1.1f%%",
        startangle=90,
        counterclock=False,
        wedgeprops={"edgecolor": "white"}
    )
    ax.set_title(f"{anrede} (n = {len(teil):,})".replace(",", "."), fontweight="bold")

fig.suptitle("Altersverteilung nach Anrede", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
tabelle_abt = pd.crosstab(gehalt_bereinigt["Abteilung"], gehalt_bereinigt["Anrede"])
print(tabelle_abt)

farben = {"Frau": "#E4572E", "Herr": "#2E86AB", "Divers": "#76B041"}
farben_liste = [farben[spalte] for spalte in tabelle_abt.columns]

#ax = achsensystem
ax = tabelle_abt.plot(kind="bar", figsize=(11, 6), color=farben_liste, width=0.8, edgecolor="white")

for container in ax.containers:
    ax.bar_label(container, fmt="{:,.0f}".format, fontsize=8, padding=2)

ax.set_title("Mitarbeiterzahl je Abteilung nach Anrede", fontsize=14, fontweight="bold")
ax.set_xlabel("Abteilung")
ax.set_ylabel("Anzahl Mitarbeiter")
ax.tick_params(axis="x", rotation=30)

ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(title="Anrede", frameon=False)

plt.tight_layout()
plt.show()

*ToDo:*
1. Zufallsketten im Namen DONE
2. Verbesserte Geburtstagsüberprüfung (16-70) DONE
2.1 Vielleicht inkorrekte Anrede filter?
3. Code Commentary verbessern
4. Komplett leere Felder DONE
5. Notiz: uns fehlen ja noch Felder, eventuell muss die Gehaltsformel angepasst werden?
6. Die Diagramme überprüfen (Anzahl Anreden, Stichproben darstellen) und kommentieren

## e) Kennzahlen (Alter und Jahresgehalt, gesamt / Herr / Frau)

## f) Zusammenhangsanalyse: Pearson- und Spearman-Korrelation Alter/Jahresgehalt

## g) Diskussion: Erste Vermutung zum Gehaltsunterschied
*(Freitext-Antwort als Markdown-Zelle)*

*Ihre Antwort hier...*

---
# Teil 2 – Unternehmensentwicklung

## a) Daten einlesen und Streudiagramm

In [ ]:
umsatz = pd.read_csv(pfad + f"nanuag_umsatz_G{g}_V{v}.txt", sep=r'\s+',
                      decimal=',', header=None, names=['Jahr','Umsatz_Mio_EUR'])
umsatz.head()

## b) Regressionsanalyse (5 Funktionstypen gemäß Anhang A – abhängig von gerader/ungerader Gruppennummer)

## c) Kreuzvalidierung (Training 1994–2018, Test 2019–2026): R²/RMSE je Modell auf Testdaten

## d) Modellwahl und Begründung (basierend auf b und c) + Residuenplot

## e) Diagramm mit optimaler Regressionskurve

## f) Prognose 2030 / 2035, Vergleich mit Polynom 6. Grades, Diskussion Extrapolation

---
# Teil 3 – Firmenlotterie „NANU-Glück“

## a) Erwartungswert und Standardabweichung von $X_{10}$ (vollständige Rechnung)

## b) $X_9, \dots, X_2$ tabellarisch

## c) Optimale Anzahl angekreuzter Zahlen + Diskussion Risiko vs. Erwartungswert

## d) Individuelle Aufgabe

In [ ]:
with open(pfad + f"nanuag_lotterie_frage_G{g}_V{v}.txt", encoding='utf-8') as f:
    print(f.read())

## e) Kombinationsspiel: $X_{10}+X_2$ gleichzeitig — Unabhängigkeitsdiskussion, Erwartungswert und Standardabweichung

---
# Teil 4 – Qualitätskontrolle in der Fertigung

## a) Daten einlesen und Kennzahlen (MR1, MR2, MR3)

In [ ]:
mr1 = pd.read_csv(pfad + f"nanuag_qc_G{g}_V{v}_MR1.txt", sep=r'\s+', decimal=',', header=None, names=['MR1'])
mr2 = pd.read_csv(pfad + f"nanuag_qc_G{g}_V{v}_MR2.txt", sep=r'\s+', decimal=',', header=None, names=['MR2'])
mr3 = pd.read_csv(pfad + f"nanuag_qc_G{g}_V{v}_MR3.txt", sep=r'\s+', decimal=',', header=None, names=['MR3'])
mr1.describe(), mr2.describe(), mr3.describe()

## b) Histogramme (mit begründeter Klassenwahl, z. B. Sturges / Freedman-Diaconis)

## c) Normalverteilung? (Histogramm + Q-Q-Plot oder Schiefe/Kurtosis)

## d) Dichtekurve für normalverteilte Messreihen einzeichnen + Skalierungsfaktor erläutern

## e) Formaler Normalverteilungstest (Shapiro-Wilk oder Kolmogorov-Smirnov)
*Hinweis: `scipy.stats.shapiro` ist für n > 5000 nicht mehr zuverlässig — ziehen Sie eine
Zufallsstichprobe (z. B. n=2000) und diskutieren Sie „statistische vs. praktische Signifikanz“
bei sehr großen Stichproben.*

---
# Teil 5 – Statistische Tests: Gehalt, Führung und Abteilungszugehörigkeit

## a) Gauß-Test (Einstichproben-Z-Test): MR2 gegen Sollmaß μ0=883,0, bekannt: σ0=1,8

## b) Zweistichproben-t-Test (Jahresgehalt Herr vs. Frau)

## c) Binomialtest (Frauenanteil Führungspositionen, $H_0: p=0{,}30$)

## d) Chi-Quadrat-Unabhängigkeitstest (Kreuztabelle Anrede × Abteilung)

## e) Mehrfachtestproblem: Bonferroni-Korrektur über die vier Tests a)–d) ($\alpha'=\alpha/4$)

## f) Management-Summary (max. 1 Seite)

*Ihre Zusammenfassung hier...*